# CHB24 Data Preparation for Time-to-Seizure Forecasting

**Purpose.** Download and preprocess the CHB24 subject (CHB-MIT Scalp EEG
Database) into a feature representation that is structurally and
numerically compatible with the already-verified CHB preprocessing in
`EEG_preprocessing.ipynb`. CHB24 is added because CHB01 (currently the only
processed CHB subject) contains only one seizure event, insufficient for a
time-to-seizure forecasting experiment.

**This notebook does NOT:**
- train a TCN, CNN/EEGNet, or any model,
- construct the final 4-class time-to-seizure target,
- download the full CHB-MIT corpus or any non-seizure subject,
- modify `EEG_preprocessing.ipynb` or its outputs.

**Reuse policy.** No shared `.py` module exists in this repository (verified:
no `src/` or similar package). All preprocessing logic below is copied
**verbatim** from the verified cells of `EEG_preprocessing.ipynb` (channel
selection, filtering, epoching, Welch-PSD feature extraction, CHB summary
parsing, seizure labeling, temporal metadata) rather than reimplemented, to
guarantee the CHB24 representation matches CHB01 exactly. Any function body
below that is new (download, manifest verification, forecasting-horizon
metadata, validation) is explicitly marked as new in its cell.

**Source of truth for parameters.** `outputs/preprocessing_metadata.json`
(written by `EEG_preprocessing.ipynb`) is loaded and asserted against, rather
than re-declaring filter/epoch constants blindly.


In [1]:
print("=" * 60)
print("CHB24 DATA PREPARATION FOR TIME-TO-SEIZURE FORECASTING")
print("=" * 60)

import gc
import json
import re
import urllib.request
import urllib.error
from pathlib import Path

import numpy as np
import pandas as pd
import mne
from scipy.signal import welch

print("\nLibrary versions:")
print("  numpy:", np.__version__)
print("  pandas:", pd.__version__)
print("  mne:", mne.__version__)

repo_root = Path("/home/evgenija/projects/EEG_timeseries_project")
data_root = repo_root / "data"
output_dir = repo_root / "outputs"
chb_dir = data_root / "raw" / "chbmit"   # SAME directory as CHB01 (existing convention)
chb_dir.mkdir(parents=True, exist_ok=True)

print(f"\nrepo_root : {repo_root}")
print(f"chb_dir   : {chb_dir}  (existing CHB01 files also live here)")
print(f"output_dir: {output_dir}")


CHB24 DATA PREPARATION FOR TIME-TO-SEIZURE FORECASTING



Library versions:
  numpy: 2.5.2
  pandas: 3.0.5
  mne: 1.12.1

repo_root : /home/evgenija/projects/EEG_timeseries_project
chb_dir   : /home/evgenija/projects/EEG_timeseries_project/data/raw/chbmit  (existing CHB01 files also live here)
output_dir: /home/evgenija/projects/EEG_timeseries_project/outputs


In [2]:
print("\n" + "=" * 60)
print("STEP 1: INSPECT EXISTING VERIFIED CHB PIPELINE (CHB01)")
print("=" * 60)

# Load the artifacts EEG_preprocessing.ipynb already produced for CHB01.
# These are the ground truth this notebook must match, not a re-derivation.

existing_metadata_path = output_dir / "preprocessing_metadata.json"
existing_chb_parquet_path = output_dir / "features_chb.parquet"

if not existing_metadata_path.exists():
    raise FileNotFoundError(
        f"{existing_metadata_path} not found -- EEG_preprocessing.ipynb must "
        f"be run first; CHB24 compatibility cannot be established without it."
    )
if not existing_chb_parquet_path.exists():
    raise FileNotFoundError(f"{existing_chb_parquet_path} not found")

with open(existing_metadata_path) as f:
    existing_metadata = json.load(f)

existing_chb_df = pd.read_parquet(existing_chb_parquet_path)

# Established preprocessing invariants (CLAUDE.md) -- asserted against the
# verified metadata rather than re-declared independently.
FILTER_LOW_HZ = existing_metadata["preprocessing_params"]["filter_low_hz"]
FILTER_HIGH_HZ = existing_metadata["preprocessing_params"]["filter_high_hz"]
EPOCH_DURATION_SEC = existing_metadata["preprocessing_params"]["epoch_duration_sec"]
EPOCH_OVERLAP = existing_metadata["preprocessing_params"]["epoch_overlap"]
expected_chb_sfreq = existing_metadata["sampling_rates_hz"]["chb"]

assert FILTER_LOW_HZ == 0.5 and FILTER_HIGH_HZ == 40.0, "Unexpected CHB filter params"
assert EPOCH_DURATION_SEC == 30.0 and EPOCH_OVERLAP == 0.0, "Unexpected CHB epoch params"

print(f"\nVerified CHB01 preprocessing params (from {existing_metadata_path.name}):")
print(f"  filter        : {FILTER_LOW_HZ}-{FILTER_HIGH_HZ} Hz")
print(f"  epoch duration: {EPOCH_DURATION_SEC}s, overlap={EPOCH_OVERLAP}")
print(f"  chb sfreq     : {expected_chb_sfreq} Hz")

print(f"\nExisting CHB01 feature table (features_chb.parquet):")
print(f"  shape  : {existing_chb_df.shape}")
print(f"  columns: {list(existing_chb_df.columns)}")
print(f"  recording_id values: {sorted(existing_chb_df['recording_id'].unique())}")

qeeg_feature_cols = [
    "std", "var", "rms", "delta", "theta", "alpha", "beta",
    "rel_delta", "rel_theta", "rel_alpha", "rel_beta",
    "theta_alpha_ratio", "delta_beta_ratio", "entropy",
]
missing_expected = [c for c in qeeg_feature_cols if c not in existing_chb_df.columns]
if missing_expected:
    raise ValueError(
        f"Existing CHB feature table is missing expected qEEG columns: "
        f"{missing_expected}. Repository reality differs from planning -- "
        f"stop and report rather than assuming the 14-feature set."
    )
print(f"\n[OK] Confirmed {len(qeeg_feature_cols)} qEEG feature columns present in existing CHB01 table")



STEP 1: INSPECT EXISTING VERIFIED CHB PIPELINE (CHB01)

Verified CHB01 preprocessing params (from preprocessing_metadata.json):
  filter        : 0.5-40.0 Hz
  epoch duration: 30.0s, overlap=0.0
  chb sfreq     : 256.0 Hz

Existing CHB01 feature table (features_chb.parquet):
  shape  : (360, 28)
  columns: ['std', 'var', 'rms', 'delta', 'theta', 'alpha', 'beta', 'rel_delta', 'rel_theta', 'rel_alpha', 'rel_beta', 'theta_alpha_ratio', 'delta_beta_ratio', 'entropy', 'recording_id', 'epoch_idx', 't_start_sec', 't_center_sec', 't_end_sec', 'y_seizure', 'std_scaled', 'var_scaled', 'rms_scaled', 'delta_scaled', 'theta_scaled', 'alpha_scaled', 'beta_scaled', 'entropy_scaled']
  recording_id values: ['chb_s01_r01', 'chb_s01_r02', 'chb_s01_r03']

[OK] Confirmed 14 qEEG feature columns present in existing CHB01 table


In [3]:
print("\n" + "=" * 60)
print("STEP 2: DOWNLOAD + PARSE chb24-summary.txt (SOURCE OF TRUTH)")
print("=" * 60)

# NEW logic (not present in EEG_preprocessing.ipynb): the summary file itself
# must be fetched, and ALL "File Name" entries in it enumerated, rather than
# trusting the hardcoded seizure-file list from project planning blindly.

CHB24_BASE_URL = "https://physionet.org/files/chbmit/1.0.0/chb24/"
chb24_summary_path = chb_dir / "chb24-summary.txt"

def download_if_missing(url, dest_path):
    """Idempotent download: skip if dest_path already exists."""
    if dest_path.exists():
        size_mb = dest_path.stat().st_size / 1e6
        print(f"  [SKIP]      {dest_path.name:20s} already exists ({size_mb:.1f} MB)")
        return "skipped"
    try:
        print(f"  [DOWNLOAD]  {dest_path.name:20s} <- {url}")
        urllib.request.urlretrieve(url, dest_path)
        size_mb = dest_path.stat().st_size / 1e6
        print(f"  [OK]        {dest_path.name:20s} ({size_mb:.1f} MB)")
        return "downloaded"
    except (urllib.error.URLError, urllib.error.HTTPError) as e:
        if dest_path.exists():
            dest_path.unlink()  # remove partial file
        raise RuntimeError(f"FAILED to download {url}: {e}") from e

download_if_missing(CHB24_BASE_URL + "chb24-summary.txt", chb24_summary_path)

summary_text = chb24_summary_path.read_text()

# Enumerate every File Name block and its declared seizure count. This is
# independent of the hardcoded planning list -- it is the actual ground truth.
blocks = re.split(r"(?=File Name: )", summary_text)
all_files_seizure_counts = {}
for block in blocks:
    name_match = re.search(r"File Name:\s*(\S+)", block)
    if not name_match:
        continue
    fname = name_match.group(1)
    n_match = re.search(r"Number of Seizures in File:\s*(\d+)", block)
    if not n_match:
        raise ValueError(f"{fname}: could not find seizure count in summary")
    all_files_seizure_counts[fname] = int(n_match.group(1))

verified_seizure_files = sorted(
    fname for fname, n in all_files_seizure_counts.items() if n > 0
)
total_seizure_events_declared = sum(all_files_seizure_counts.values())

print(f"\nFiles listed in chb24-summary.txt: {len(all_files_seizure_counts)}")
print(f"Files with >=1 declared seizure  : {len(verified_seizure_files)}")
print(f"Total declared seizure events    : {total_seizure_events_declared}")
for fname in verified_seizure_files:
    print(f"  {fname}: {all_files_seizure_counts[fname]} seizure(s)")

# Compare against project planning's hardcoded list (do not trust it blindly).
planned_seizure_recordings = [
    "chb24_01", "chb24_03", "chb24_04", "chb24_06", "chb24_07", "chb24_09",
    "chb24_11", "chb24_13", "chb24_14", "chb24_15", "chb24_17", "chb24_21",
]
planned_files = sorted(f"{r}.edf" for r in planned_seizure_recordings)

print(f"\nPlanned (hardcoded) list has {len(planned_files)} files.")
if planned_files == verified_seizure_files:
    print("[OK] Verified summary file EXACTLY matches the planned hardcoded list.")
else:
    only_planned = sorted(set(planned_files) - set(verified_seizure_files))
    only_verified = sorted(set(verified_seizure_files) - set(planned_files))
    print("[WARNING] DISCREPANCY between planned list and verified summary file:")
    print(f"  in planned list but NOT actually seizure-containing: {only_planned}")
    print(f"  actually seizure-containing but NOT in planned list: {only_verified}")
    print("Proceeding with the VERIFIED list (summary file is source of truth).")

# The verified list is authoritative for everything downstream.
chb24_recording_ids = [Path(f).stem for f in verified_seizure_files]  # e.g. "chb24_01"
print(f"\n[OK] Using {len(chb24_recording_ids)} verified seizure-containing recordings")



STEP 2: DOWNLOAD + PARSE chb24-summary.txt (SOURCE OF TRUTH)
  [SKIP]      chb24-summary.txt    already exists (0.0 MB)

Files listed in chb24-summary.txt: 12
Files with >=1 declared seizure  : 12
Total declared seizure events    : 16
  chb24_01.edf: 2 seizure(s)
  chb24_03.edf: 2 seizure(s)
  chb24_04.edf: 3 seizure(s)
  chb24_06.edf: 1 seizure(s)
  chb24_07.edf: 1 seizure(s)
  chb24_09.edf: 1 seizure(s)
  chb24_11.edf: 1 seizure(s)
  chb24_13.edf: 1 seizure(s)
  chb24_14.edf: 1 seizure(s)
  chb24_15.edf: 1 seizure(s)
  chb24_17.edf: 1 seizure(s)
  chb24_21.edf: 1 seizure(s)

Planned (hardcoded) list has 12 files.
[OK] Verified summary file EXACTLY matches the planned hardcoded list.

[OK] Using 12 verified seizure-containing recordings


In [4]:
print("\n" + "=" * 60)
print("STEP 3: DOWNLOAD SEIZURE-CONTAINING CHB24 EDFs")
print("=" * 60)

# Idempotent, deterministic, restartable: each file is checked individually
# before download. Only the verified seizure-containing recordings are
# fetched -- no full-corpus or non-seizure-subject download.

chb24_files = {
    rec_id: chb_dir / f"{rec_id}.edf" for rec_id in chb24_recording_ids
}

download_results = {"downloaded": [], "skipped": [], "failed": []}
total_bytes = 0

for rec_id, dest_path in chb24_files.items():
    url = CHB24_BASE_URL + dest_path.name
    try:
        status = download_if_missing(url, dest_path)
        download_results[status].append(rec_id)
        total_bytes += dest_path.stat().st_size
    except RuntimeError as e:
        print(f"  [FAIL] {e}")
        download_results["failed"].append(rec_id)

print(f"\nRequested files : {len(chb24_files)}")
print(f"Downloaded      : {len(download_results['downloaded'])} {download_results['downloaded']}")
print(f"Already present : {len(download_results['skipped'])} {download_results['skipped']}")
print(f"Failed          : {len(download_results['failed'])} {download_results['failed']}")
print(f"Total size on disk: {total_bytes / 1e6:.1f} MB")

if download_results["failed"]:
    raise RuntimeError(
        f"Failed to obtain required EDFs: {download_results['failed']}. "
        f"Cannot proceed with an incomplete/corrupted seizure set."
    )

print("\n[OK] All required CHB24 seizure EDFs are available on disk")



STEP 3: DOWNLOAD SEIZURE-CONTAINING CHB24 EDFs
  [SKIP]      chb24_01.edf         already exists (42.4 MB)
  [SKIP]      chb24_03.edf         already exists (42.4 MB)
  [SKIP]      chb24_04.edf         already exists (42.4 MB)
  [SKIP]      chb24_06.edf         already exists (42.4 MB)
  [SKIP]      chb24_07.edf         already exists (42.4 MB)
  [SKIP]      chb24_09.edf         already exists (42.4 MB)
  [SKIP]      chb24_11.edf         already exists (42.4 MB)
  [SKIP]      chb24_13.edf         already exists (42.4 MB)
  [SKIP]      chb24_14.edf         already exists (42.4 MB)
  [SKIP]      chb24_15.edf         already exists (42.4 MB)
  [SKIP]      chb24_17.edf         already exists (42.4 MB)
  [SKIP]      chb24_21.edf         already exists (42.4 MB)

Requested files : 12
Downloaded      : 0 []
Already present : 12 ['chb24_01', 'chb24_03', 'chb24_04', 'chb24_06', 'chb24_07', 'chb24_09', 'chb24_11', 'chb24_13', 'chb24_14', 'chb24_15', 'chb24_17', 'chb24_21']
Failed          : 0 [

In [5]:
print("\n" + "=" * 60)
print("STEP 4: VERIFY EDF METADATA AND MONTAGE (BEFORE PROCESSING)")
print("=" * 60)

# Lightweight (preload=False) header-only inspection: sampling frequency,
# channel names/count, duration. Compares against the verified CHB01 montage
# rather than assuming equivalence.

existing_chb01_summary_path = chb_dir / "chb01-summary.txt"
chb01_channels_declared = []
if existing_chb01_summary_path.exists():
    txt = existing_chb01_summary_path.read_text()
    chb01_channels_declared = re.findall(r"Channel \d+:\s*(\S+)", txt)

chb24_channels_declared = re.findall(r"Channel \d+:\s*(\S+)", summary_text)
declared_sfreq_match = re.search(r"Data Sampling Rate:\s*([\d.]+)\s*Hz", summary_text)
chb24_declared_sfreq = float(declared_sfreq_match.group(1)) if declared_sfreq_match else None

print(f"CHB24 declared sampling rate (summary file): {chb24_declared_sfreq} Hz")
print(f"CHB24 declared channel count (summary file) : {len(chb24_channels_declared)}")
if chb01_channels_declared:
    montage_match = chb24_channels_declared == chb01_channels_declared
    print(f"CHB24 vs CHB01 declared montage identical    : {montage_match}")
    if not montage_match:
        print(f"  CHB01: {chb01_channels_declared}")
        print(f"  CHB24: {chb24_channels_declared}")

print("\nPer-file EDF header verification (preload=False, no data loaded):")
edf_metadata = {}
for rec_id, path in chb24_files.items():
    raw_header = mne.io.read_raw_edf(path, preload=False, verbose=False)
    sfreq = float(raw_header.info["sfreq"])
    ch_names = list(raw_header.ch_names)
    # Total recorded duration = n_samples / sfreq. raw.times[-1] is the
    # timestamp of the LAST sample ((n-1)/sfreq), not the total duration --
    # using it here would understate duration by one sample period and can
    # produce false boundary-violation failures for recordings whose sample
    # count divides evenly into the epoch window.
    duration_sec = float(raw_header.n_times) / sfreq
    edf_metadata[rec_id] = {"sfreq": sfreq, "ch_names": ch_names, "duration_sec": duration_sec}
    print(f"  {rec_id:12s}: sfreq={sfreq:6.1f}Hz, channels={len(ch_names):2d}, "
          f"duration={duration_sec:7.1f}s ({duration_sec/3600:.2f}h)")
    del raw_header

# Verify sfreq is uniform and matches the CHB01-verified sampling rate.
sfreqs_found = {m["sfreq"] for m in edf_metadata.values()}
if len(sfreqs_found) != 1:
    raise ValueError(f"CHB24 recordings have inconsistent sampling rates: {sfreqs_found}")
chb24_sfreq = sfreqs_found.pop()
if not np.isclose(chb24_sfreq, expected_chb_sfreq):
    raise ValueError(
        f"CHB24 sfreq ({chb24_sfreq} Hz) does not match verified CHB01 sfreq "
        f"({expected_chb_sfreq} Hz) -- datasets would not be comparable."
    )
print(f"\n[OK] Uniform sfreq across CHB24 recordings: {chb24_sfreq} Hz (matches CHB01)")

# Verify channel names are identical across all CHB24 recordings.
ch_name_sets = {tuple(m["ch_names"]) for m in edf_metadata.values()}
if len(ch_name_sets) != 1:
    print("[WARNING] CHB24 recordings do NOT all share identical channel names/order.")
    for rec_id, m in edf_metadata.items():
        print(f"  {rec_id}: {m['ch_names']}")
    montage_uniform_across_recordings = False
else:
    montage_uniform_across_recordings = True
    print(f"[OK] Identical channel set/order across all {len(edf_metadata)} CHB24 recordings")

chb24_actual_channels = ch_name_sets.copy().pop() if montage_uniform_across_recordings else None
if chb24_actual_channels is not None and chb01_channels_declared:
    if list(chb24_actual_channels) == chb01_channels_declared:
        print("[OK] CHB24 actual EDF channel names match CHB01's declared montage exactly")
    else:
        print("[WARNING] CHB24 actual EDF channel names DIFFER from CHB01's declared montage:")
        print(f"  CHB01: {chb01_channels_declared}")
        print(f"  CHB24: {list(chb24_actual_channels)}")



STEP 4: VERIFY EDF METADATA AND MONTAGE (BEFORE PROCESSING)
CHB24 declared sampling rate (summary file): 256.0 Hz
CHB24 declared channel count (summary file) : 23
CHB24 vs CHB01 declared montage identical    : True

Per-file EDF header verification (preload=False, no data loaded):


  chb24_01    : sfreq= 256.0Hz, channels=23, duration= 3600.0s (1.00h)
  chb24_03    : sfreq= 256.0Hz, channels=23, duration= 3600.0s (1.00h)
  chb24_04    : sfreq= 256.0Hz, channels=23, duration= 3600.0s (1.00h)
  chb24_06    : sfreq= 256.0Hz, channels=23, duration= 3600.0s (1.00h)
  chb24_07    : sfreq= 256.0Hz, channels=23, duration= 3600.0s (1.00h)
  chb24_09    : sfreq= 256.0Hz, channels=23, duration= 3600.0s (1.00h)
  chb24_11    : sfreq= 256.0Hz, channels=23, duration= 3600.0s (1.00h)
  chb24_13    : sfreq= 256.0Hz, channels=23, duration= 3600.0s (1.00h)
  chb24_14    : sfreq= 256.0Hz, channels=23, duration= 3600.0s (1.00h)
  chb24_15    : sfreq= 256.0Hz, channels=23, duration= 3600.0s (1.00h)
  chb24_17    : sfreq= 256.0Hz, channels=23, duration= 3600.0s (1.00h)
  chb24_21    : sfreq= 256.0Hz, channels=23, duration= 3600.0s (1.00h)

[OK] Uniform sfreq across CHB24 recordings: 256.0 Hz (matches CHB01)
[OK] Identical channel set/order across all 12 CHB24 recordings
[WARNING] CHB2

/tmp/ipykernel_38983/3029005001.py:31: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw_header = mne.io.read_raw_edf(path, preload=False, verbose=False)
/tmp/ipykernel_38983/3029005001.py:31: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw_header = mne.io.read_raw_edf(path, preload=False, verbose=False)
/tmp/ipykernel_38983/3029005001.py:31: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw_header = mne.io.read_raw_edf(path, preload=False, verbose=False)
/tmp/ipykernel_38983/3029005001.py:31: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw_header = mne.io.read_raw_edf(path, preload=False, verbose=False)
/tmp/ipykernel_38983/3029005001.py:31: RuntimeWarning: Channel names are not uni

In [6]:
print("\n" + "=" * 60)
print("STEP 5: VERIFIED PREPROCESSING HELPERS (copied verbatim, not reimplemented)")
print("=" * 60)

# Everything in this cell is copied VERBATIM from the verified cells of
# EEG_preprocessing.ipynb (Steps 6, 7, 8, 10, 10B). No repository shared
# module exists to import from, so verbatim duplication is the safest way
# to guarantee CHB24 is processed identically to CHB01. Do not modify this
# cell independently of the source notebook.

def preprocess_eeg(raw, dataset_name, l_freq=FILTER_LOW_HZ, h_freq=FILTER_HIGH_HZ):
    """
    Select EEG-typed channels, apply band-pass filter, validate.

    Order: correct channel types (done upstream for Sleep) -> select EEG
    -> filter -> validate -> convert to NumPy. This avoids converting to
    NumPy before MNE-level corrections are applied.

    Returns:
        eeg_raw: MNE Raw object (EEG channels only, filtered)
        data: NumPy array (channels, samples)
        sfreq: Sampling frequency
    """
    eeg_raw = raw.copy().pick(picks="eeg")

    if len(eeg_raw.ch_names) == 0:
        raise ValueError(f"{dataset_name}: No EEG channels found")

    sfreq = float(eeg_raw.info["sfreq"])
    nyquist = sfreq / 2.0

    if h_freq >= nyquist:
        raise ValueError(
            f"{dataset_name}: filter cutoff {h_freq}Hz >= Nyquist {nyquist}Hz "
            f"(sfreq={sfreq}Hz)"
        )

    eeg_raw.filter(l_freq=l_freq, h_freq=h_freq, verbose=False)

    data = eeg_raw.get_data()

    if data.ndim != 2:
        raise ValueError(f"{dataset_name}: Expected 2D data, got {data.shape}")

    if not np.isfinite(data).all():
        raise ValueError(f"{dataset_name}: Data contains NaN/Inf after filtering")

    print(f"{dataset_name:16s}: channels={len(eeg_raw.ch_names):2d}, "
          f"sfreq={sfreq:6.1f}Hz, shape={data.shape}, "
          f"filter=[{l_freq:.1f}, {h_freq:.1f}]Hz")

    return eeg_raw, data, sfreq


def make_epochs(data, sfreq, win_sec=EPOCH_DURATION_SEC, overlap=EPOCH_OVERLAP):
    """
    Create fixed-duration, non-overlapping epochs from continuous data.

    Temporal convention (established, CLAUDE.md):
        +1 epoch = 30 s, +2 epochs = 60 s, +4 epochs = 120 s.
    Zero overlap is deliberate. Leftover samples that don't fill a full
    window are dropped (not zero-padded), so all epochs have equal, real
    signal content.

    Parameters:
        data: (channels, samples)
        sfreq: Sampling frequency in Hz
        win_sec: Window duration in seconds
        overlap: Overlap fraction [0, 1)

    Returns:
        epochs: (n_epochs, channels, win_samples)
    """
    if data.ndim != 2:
        raise ValueError(f"Expected 2D data, got {data.shape}")

    if not (0.0 <= overlap < 1.0):
        raise ValueError("overlap must be in [0, 1)")

    win_samp = int(round(win_sec * sfreq))
    step = int(round(win_samp * (1 - overlap)))

    if win_samp <= 0:
        raise ValueError("win_sec and sfreq must produce positive window size")

    if step <= 0:
        raise ValueError("overlap is too large")

    n_samples = data.shape[1]

    if n_samples < win_samp:
        raise ValueError(
            f"Recording has {n_samples} samples, but epoch requires {win_samp}"
        )

    starts = np.arange(0, n_samples - win_samp + 1, step)

    epochs = np.stack(
        [data[:, start:start + win_samp] for start in starts],
        axis=0
    )

    return epochs


def validate_epochs(name, epochs, sfreq, win_sec=EPOCH_DURATION_SEC,
                   expected_channels=None):
    """Validate epoch structure and content (shape, channels, finiteness)."""
    expected_samples = int(round(win_sec * sfreq))

    if epochs.ndim != 3:
        raise ValueError(f"{name}: Expected 3D, got {epochs.shape}")

    if epochs.shape[2] != expected_samples:
        raise ValueError(
            f"{name}: Expected {expected_samples} samples/epoch, got {epochs.shape[2]}"
        )

    if expected_channels is not None:
        if epochs.shape[1] != expected_channels:
            raise ValueError(
                f"{name}: Expected {expected_channels} channels, got {epochs.shape[1]}"
            )

    if not np.isfinite(epochs).all():
        raise ValueError(f"{name}: Contains NaN/Inf values")

    print(f"{name:16s}: epochs={epochs.shape[0]:5d}, channels={epochs.shape[1]:2d}, "
          f"samples={epochs.shape[2]:5d}, duration={win_sec:.0f}s")


def bandpower(x, sfreq, fmin, fmax, nperseg=None):
    """Absolute band power in [fmin, fmax] via Welch PSD (trapezoidal integration)."""
    if nperseg is None:
        nperseg = min(len(x), int(2 * sfreq))  # ~2s sub-windows for PSD estimation
    freqs, psd = welch(x, fs=sfreq, nperseg=nperseg)
    mask = (freqs >= fmin) & (freqs <= fmax)
    if not np.any(mask):
        return np.nan
    return np.trapezoid(psd[mask], freqs[mask])


def spectral_entropy_welch(x, sfreq, fmin=1.0, fmax=40.0, nperseg=None, eps=1e-12):
    """Shannon entropy of the normalized Welch PSD, normalized to [0, 1]."""
    if nperseg is None:
        nperseg = min(len(x), int(2 * sfreq))
    freqs, psd = welch(x, fs=sfreq, nperseg=nperseg)
    mask = (freqs >= fmin) & (freqs <= fmax)
    psd_band = np.maximum(psd[mask], eps)
    p = psd_band / psd_band.sum()
    ent = -np.sum(p * np.log(p))
    return ent / np.log(len(p))


def extract_features(epochs, sfreq):
    """
    Extract qEEG spectral features from epochs, averaged across channels.

    Bands: delta 0.5-4Hz, theta 4-8Hz, alpha 8-13Hz, beta 13-30Hz.
    """
    features = []
    eps = 1e-12

    for ep in epochs:
        std = np.std(ep)
        var = np.var(ep)
        rms = np.sqrt(np.mean(ep**2))

        delta = np.mean([bandpower(ch, sfreq, 0.5, 4) for ch in ep])
        theta = np.mean([bandpower(ch, sfreq, 4, 8) for ch in ep])
        alpha = np.mean([bandpower(ch, sfreq, 8, 13) for ch in ep])
        beta = np.mean([bandpower(ch, sfreq, 13, 30) for ch in ep])

        total_power = delta + theta + alpha + beta + eps
        rel_delta = delta / total_power
        rel_theta = theta / total_power
        rel_alpha = alpha / total_power
        rel_beta = beta / total_power

        theta_alpha_ratio = theta / (alpha + eps)
        delta_beta_ratio = delta / (beta + eps)

        ent = np.mean([spectral_entropy_welch(ch, sfreq) for ch in ep])

        features.append({
            "std": std, "var": var, "rms": rms,
            "delta": delta, "theta": theta, "alpha": alpha, "beta": beta,
            "rel_delta": rel_delta, "rel_theta": rel_theta,
            "rel_alpha": rel_alpha, "rel_beta": rel_beta,
            "theta_alpha_ratio": theta_alpha_ratio,
            "delta_beta_ratio": delta_beta_ratio,
            "entropy": ent,
        })

    return pd.DataFrame(features)


def parse_chb_summary(summary_path, known_edf_filenames):
    """
    Parse a CHB-MIT *-summary.txt file into {edf_filename: [(start_s, end_s), ...]}.

    Only entries whose File Name is in `known_edf_filenames` are kept.
    Raises if a file's declared seizure count doesn't match the number of
    Start/End time pairs actually parsed for it, or if any known
    recording has no summary entry at all.
    """
    text = summary_path.read_text()
    blocks = re.split(r"(?=File Name: )", text)

    annotations = {}
    for block in blocks:
        name_match = re.search(r"File Name:\s*(\S+)", block)
        if not name_match:
            continue
        fname = name_match.group(1)
        if fname not in known_edf_filenames:
            continue

        n_match = re.search(r"Number of Seizures in File:\s*(\d+)", block)
        if not n_match:
            raise ValueError(f"{fname}: could not find seizure count in summary")
        n_seizures = int(n_match.group(1))

        starts = [int(s) for s in re.findall(r"Seizure Start Time:\s*(\d+)\s*seconds", block)]
        ends = [int(s) for s in re.findall(r"Seizure End Time:\s*(\d+)\s*seconds", block)]

        if len(starts) != n_seizures or len(ends) != n_seizures:
            raise ValueError(
                f"{fname}: declared {n_seizures} seizures but parsed "
                f"{len(starts)} start / {len(ends)} end times"
            )

        annotations[fname] = list(zip(starts, ends))

    missing = set(known_edf_filenames) - set(annotations)
    if missing:
        raise ValueError(f"No summary entry found for: {missing}")

    return annotations


def label_seizure_epochs(t_start_sec, t_end_sec, intervals):
    """
    y_seizure = 1 iff [t_start_sec, t_end_sec) intersects any annotated
    seizure interval, using true interval intersection:
        epoch_start < seizure_end AND epoch_end > seizure_start
    (not epoch-center membership).
    """
    y = np.zeros(len(t_start_sec), dtype=int)
    for i, (t_start, t_end) in enumerate(zip(t_start_sec, t_end_sec)):
        for seiz_start, seiz_end in intervals:
            if t_start < seiz_end and t_end > seiz_start:
                y[i] = 1
                break
    return y


def add_temporal_metadata(df, win_sec=EPOCH_DURATION_SEC):
    """
    Add per-recording epoch index and elapsed-time columns.

    epoch_idx restarts at 0 for every recording (groupby recording_id) so
    that time is always relative to that recording's own start.
    """
    df = df.copy()

    if "recording_id" not in df.columns:
        raise ValueError("Expected 'recording_id' column")

    df["epoch_idx"] = df.groupby("recording_id").cumcount()
    df["t_start_sec"] = df["epoch_idx"] * win_sec
    df["t_center_sec"] = df["t_start_sec"] + win_sec / 2.0
    df["t_end_sec"] = df["t_start_sec"] + win_sec

    return df

print("[OK] Verified helper functions loaded (verbatim copies from EEG_preprocessing.ipynb):")
for fn in [preprocess_eeg, make_epochs, validate_epochs, bandpower,
           spectral_entropy_welch, extract_features, parse_chb_summary,
           label_seizure_epochs, add_temporal_metadata]:
    print(f"  - {fn.__name__}")



STEP 5: VERIFIED PREPROCESSING HELPERS (copied verbatim, not reimplemented)
[OK] Verified helper functions loaded (verbatim copies from EEG_preprocessing.ipynb):
  - preprocess_eeg
  - make_epochs
  - validate_epochs
  - bandpower
  - spectral_entropy_welch
  - extract_features
  - parse_chb_summary
  - label_seizure_epochs
  - add_temporal_metadata


In [7]:
print("\n" + "=" * 60)
print("STEP 6: PARSE CHB24 SEIZURE INTERVALS (restricted to loaded recordings)")
print("=" * 60)

chb24_edf_filenames = {rec_id: chb24_files[rec_id].name for rec_id in chb24_recording_ids}

chb24_seizure_intervals_by_file = parse_chb_summary(
    chb24_summary_path,
    known_edf_filenames=set(chb24_edf_filenames.values()),
)
chb24_seizure_intervals = {
    rec_id: chb24_seizure_intervals_by_file[fname]
    for rec_id, fname in chb24_edf_filenames.items()
}

n_seizure_events_total = sum(len(v) for v in chb24_seizure_intervals.values())
print(f"\nParsed from {chb24_summary_path.name}:")
for rec_id, intervals in chb24_seizure_intervals.items():
    for start, end in intervals:
        print(f"  {rec_id}: seizure {start}-{end}s (duration {end-start}s)")

print(f"\n[OK] Total seizure events parsed: {n_seizure_events_total}")
if n_seizure_events_total != total_seizure_events_declared:
    raise ValueError(
        f"Parsed seizure count ({n_seizure_events_total}) != declared count "
        f"in summary ({total_seizure_events_declared})"
    )



STEP 6: PARSE CHB24 SEIZURE INTERVALS (restricted to loaded recordings)

Parsed from chb24-summary.txt:
  chb24_01: seizure 480-505s (duration 25s)
  chb24_01: seizure 2451-2476s (duration 25s)
  chb24_03: seizure 231-260s (duration 29s)
  chb24_03: seizure 2883-2908s (duration 25s)
  chb24_04: seizure 1088-1120s (duration 32s)
  chb24_04: seizure 1411-1438s (duration 27s)
  chb24_04: seizure 1745-1764s (duration 19s)
  chb24_06: seizure 1229-1253s (duration 24s)
  chb24_07: seizure 38-60s (duration 22s)
  chb24_09: seizure 1745-1764s (duration 19s)
  chb24_11: seizure 3527-3597s (duration 70s)
  chb24_13: seizure 3288-3304s (duration 16s)
  chb24_14: seizure 1939-1966s (duration 27s)
  chb24_15: seizure 3552-3569s (duration 17s)
  chb24_17: seizure 3515-3581s (duration 66s)
  chb24_21: seizure 2804-2872s (duration 68s)

[OK] Total seizure events parsed: 16


In [8]:
print("\n" + "=" * 60)
print("STEP 7: INCREMENTAL PER-EDF PROCESSING (memory-conservative)")
print("=" * 60)

# One EDF at a time: load -> preprocess -> epoch -> extract 14 qEEG features
# -> release raw/epoch arrays -> gc.collect() -> continue. No giant
# all_recordings_raw / all_epochs_raw array is ever constructed. Only a
# compact per-recording feature DataFrame is kept in memory, matching the
# feature-sequence (not raw-tensor) representation the TCN will consume.
#
# Restartable: a per-recording checkpoint parquet is written immediately
# after each EDF is processed, and skipped on rerun if already present.

checkpoint_dir = output_dir / "preprocessed_data" / "chb24_checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

SUBJECT_ID = 24
DATASET_NAME = "chb"

feat_chb24_parts = []

for rec_id in chb24_recording_ids:
    checkpoint_path = checkpoint_dir / f"features_{rec_id}.parquet"

    if checkpoint_path.exists():
        print(f"\n[CHECKPOINT SKIP] {rec_id}: {checkpoint_path.name} already exists")
        feat_chb24_parts.append(pd.read_parquet(checkpoint_path))
        continue

    print(f"\nProcessing {rec_id} ...")
    path = chb24_files[rec_id]

    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
    eeg_raw, data, sfreq = preprocess_eeg(raw, rec_id)

    if eeg_raw.ch_names != list(chb24_actual_channels):
        raise ValueError(
            f"{rec_id}: EEG channel selection ({eeg_raw.ch_names}) does not "
            f"match the verified uniform CHB24 montage."
        )

    epochs = make_epochs(data, sfreq)
    validate_epochs(rec_id, epochs, sfreq, expected_channels=len(chb24_actual_channels))

    feat = extract_features(epochs, sfreq)
    feat["recording_id"] = rec_id
    feat["dataset"] = DATASET_NAME
    feat["subject_id"] = SUBJECT_ID

    feat.to_parquet(checkpoint_path, index=False)
    feat_chb24_parts.append(feat)
    print(f"  [OK] {rec_id}: {feat.shape[0]} epochs -> checkpoint saved")

    # Release raw EEG data from memory before moving to the next EDF.
    del raw, eeg_raw, data, epochs
    gc.collect()

feat_chb24_all = pd.concat(feat_chb24_parts, ignore_index=True)

# recording_id convention: match existing "chb_s01_r01" pattern exactly.
feat_chb24_all["recording_id"] = feat_chb24_all["recording_id"].apply(
    lambda rid: f"chb_s{SUBJECT_ID:02d}_r{rid.split('_')[1]}"
)

print(f"\n[OK] All {len(chb24_recording_ids)} CHB24 recordings processed")
print(f"  Total epochs: {feat_chb24_all.shape[0]}")
print(f"  recording_id values: {sorted(feat_chb24_all['recording_id'].unique())}")



STEP 7: INCREMENTAL PER-EDF PROCESSING (memory-conservative)

[CHECKPOINT SKIP] chb24_01: features_chb24_01.parquet already exists

[CHECKPOINT SKIP] chb24_03: features_chb24_03.parquet already exists

[CHECKPOINT SKIP] chb24_04: features_chb24_04.parquet already exists

[CHECKPOINT SKIP] chb24_06: features_chb24_06.parquet already exists

[CHECKPOINT SKIP] chb24_07: features_chb24_07.parquet already exists

[CHECKPOINT SKIP] chb24_09: features_chb24_09.parquet already exists

[CHECKPOINT SKIP] chb24_11: features_chb24_11.parquet already exists

[CHECKPOINT SKIP] chb24_13: features_chb24_13.parquet already exists

[CHECKPOINT SKIP] chb24_14: features_chb24_14.parquet already exists

[CHECKPOINT SKIP] chb24_15: features_chb24_15.parquet already exists

[CHECKPOINT SKIP] chb24_17: features_chb24_17.parquet already exists

[CHECKPOINT SKIP] chb24_21: features_chb24_21.parquet already exists

[OK] All 12 CHB24 recordings processed
  Total epochs: 1440
  recording_id values: ['chb_s24_r01'

In [9]:
print("\n" + "=" * 60)
print("STEP 8: TEMPORAL METADATA + SEIZURE LABELING + FORECASTING HORIZON")
print("=" * 60)

feat_chb24_all = add_temporal_metadata(feat_chb24_all)

# Re-key seizure intervals to match the final recording_id naming
# (chb_s24_rNN), consistent with recording_id used in add_temporal_metadata.
chb24_seizure_intervals_final = {
    f"chb_s{SUBJECT_ID:02d}_r{rec_id.split('_')[1]}": intervals
    for rec_id, intervals in chb24_seizure_intervals.items()
}

feat_chb24_all["y_seizure"] = 0
for recording_id, intervals in chb24_seizure_intervals_final.items():
    mask = feat_chb24_all["recording_id"] == recording_id
    feat_chb24_all.loc[mask, "y_seizure"] = label_seizure_epochs(
        feat_chb24_all.loc[mask, "t_start_sec"],
        feat_chb24_all.loc[mask, "t_end_sec"],
        intervals,
    )

n_seizure_positive = int(feat_chb24_all["y_seizure"].sum())
n_seizure_negative = len(feat_chb24_all) - n_seizure_positive
assert set(feat_chb24_all["y_seizure"].unique()) <= {0, 1}, "y_seizure must be binary"

print(f"y_seizure labeled: {n_seizure_positive} positive / {n_seizure_negative} negative epochs")

# --- NEW: forecasting-horizon metadata (not present in EEG_preprocessing.ipynb) ---
# next_seizure_start_sec: the onset of the nearest seizure in the SAME
# recording whose onset has not yet occurred by this epoch's end
# (onset >= t_end_sec). NaN if no such future seizure exists in the
# recording. time_to_next_seizure_sec = next_seizure_start_sec - t_end_sec.
# This is purely descriptive metadata; the 4-class target and the exclusion
# of seizure-overlapping epochs are explicitly left to the future modeling
# notebook, per task scope.

def compute_time_to_next_seizure(df, seizure_intervals_by_recording):
    df = df.copy()
    df["next_seizure_start_sec"] = np.nan
    df["time_to_next_seizure_sec"] = np.nan

    for recording_id, intervals in seizure_intervals_by_recording.items():
        mask = df["recording_id"] == recording_id
        if not mask.any():
            continue
        onsets = np.sort(np.array([s for s, _ in intervals], dtype=float))
        t_end = df.loc[mask, "t_end_sec"].to_numpy()

        # For each epoch, find the first onset >= t_end (searchsorted 'left').
        idx = np.searchsorted(onsets, t_end, side="left")
        next_onset = np.where(idx < len(onsets), onsets[np.clip(idx, 0, len(onsets) - 1)], np.nan)

        df.loc[mask, "next_seizure_start_sec"] = next_onset
        df.loc[mask, "time_to_next_seizure_sec"] = next_onset - t_end

    return df

feat_chb24_all = compute_time_to_next_seizure(feat_chb24_all, chb24_seizure_intervals_final)

n_with_future_seizure = int(feat_chb24_all["next_seizure_start_sec"].notna().sum())
n_no_future_seizure = int(feat_chb24_all["next_seizure_start_sec"].isna().sum())
print(f"Epochs with a future seizure in-recording: {n_with_future_seizure}")
print(f"Epochs with NO future seizure (NaN)      : {n_no_future_seizure}")

# time_to_next_seizure_sec must be >= 0 wherever defined (by construction),
# and NaN exactly where next_seizure_start_sec is NaN (never mis-filled).
finite_ttn = feat_chb24_all["time_to_next_seizure_sec"].dropna()
assert (finite_ttn >= 0).all(), "time_to_next_seizure_sec must be non-negative"
assert feat_chb24_all["time_to_next_seizure_sec"].isna().sum() == n_no_future_seizure

print("\n[OK] Temporal metadata, y_seizure, and forecasting-horizon columns added")



STEP 8: TEMPORAL METADATA + SEIZURE LABELING + FORECASTING HORIZON


y_seizure labeled: 28 positive / 1412 negative epochs
Epochs with a future seizure in-recording: 952
Epochs with NO future seizure (NaN)      : 488

[OK] Temporal metadata, y_seizure, and forecasting-horizon columns added


In [10]:
print("\n" + "=" * 60)
print("STEP 9: SAVE CHB24 PROCESSED FEATURE TABLE")
print("=" * 60)

output_csv_path = output_dir / "features_chb24_forecasting.csv"
output_parquet_path = output_dir / "features_chb24_forecasting.parquet"

feat_chb24_all.to_csv(output_csv_path, index=False)
feat_chb24_all.to_parquet(output_parquet_path, index=False)

print(f"Saved: {output_parquet_path}")
print(f"Saved: {output_csv_path}")
print(f"Shape: {feat_chb24_all.shape}")
print(f"Columns: {list(feat_chb24_all.columns)}")



STEP 9: SAVE CHB24 PROCESSED FEATURE TABLE


Saved: /home/evgenija/projects/EEG_timeseries_project/outputs/features_chb24_forecasting.parquet
Saved: /home/evgenija/projects/EEG_timeseries_project/outputs/features_chb24_forecasting.csv
Shape: (1440, 24)
Columns: ['std', 'var', 'rms', 'delta', 'theta', 'alpha', 'beta', 'rel_delta', 'rel_theta', 'rel_alpha', 'rel_beta', 'theta_alpha_ratio', 'delta_beta_ratio', 'entropy', 'recording_id', 'dataset', 'subject_id', 'epoch_idx', 't_start_sec', 't_center_sec', 't_end_sec', 'y_seizure', 'next_seizure_start_sec', 'time_to_next_seizure_sec']


In [11]:
print("\n" + "=" * 60)
print("STEP 10: COMPATIBILITY VALIDATION vs EXISTING CHB01 TABLE")
print("=" * 60)

existing_cols = set(existing_chb_df.columns)
new_cols = set(feat_chb24_all.columns)

feature_cols_identical = set(qeeg_feature_cols).issubset(existing_cols) and \
                          set(qeeg_feature_cols).issubset(new_cols)

print(f"Existing CHB01 columns ({len(existing_cols)}): {sorted(existing_cols)}")
print(f"\nNew CHB24 columns ({len(new_cols)}): {sorted(new_cols)}")
print(f"\nqEEG feature columns identical? {feature_cols_identical}")

only_in_existing = sorted(existing_cols - new_cols)
only_in_new = sorted(new_cols - existing_cols)
print(f"\nColumns only in existing CHB01 table: {only_in_existing}")
print(f"Columns only in new CHB24 table     : {only_in_new}")
print("  (new columns are additive metadata for cross-dataset/forecasting use;")
print("   no existing column is renamed, removed, or duplicated under a new name)")

print("\ndtype comparison for shared columns:")
dtype_mismatches = []
for col in sorted(existing_cols & new_cols):
    dt_old = existing_chb_df[col].dtype
    dt_new = feat_chb24_all[col].dtype
    # pd.api.types.is_numeric_dtype (not np.issubdtype) handles pandas
    # extension dtypes (e.g. StringDtype on recording_id) without raising.
    compatible = pd.api.types.is_numeric_dtype(dt_old) == pd.api.types.is_numeric_dtype(dt_new)
    if not compatible:
        dtype_mismatches.append((col, dt_old, dt_new))
    print(f"  {col:24s} existing={str(dt_old):10s} new={str(dt_new):10s} compatible={compatible}")

feature_table_no_forecasting = feat_chb24_all.drop(
    columns=["next_seizure_start_sec", "time_to_next_seizure_sec"]
)
n_nan_new = int(feature_table_no_forecasting.isna().sum().sum())
n_inf_new = int(np.isinf(
    feature_table_no_forecasting.select_dtypes(include=[np.number])
).sum().sum())
print(f"\nMissing values in new table (excluding intentional NaN forecasting cols): {n_nan_new}")
print(f"Infinite values in new table: {n_inf_new}")

if dtype_mismatches:
    raise ValueError(f"dtype incompatibilities found: {dtype_mismatches}")
if n_inf_new != 0:
    raise ValueError(f"CHB24 feature table contains {n_inf_new} infinite values")

print("\n[OK] CHB24 table is schema- and dtype-compatible with the existing CHB01 table")



STEP 10: COMPATIBILITY VALIDATION vs EXISTING CHB01 TABLE
Existing CHB01 columns (28): ['alpha', 'alpha_scaled', 'beta', 'beta_scaled', 'delta', 'delta_beta_ratio', 'delta_scaled', 'entropy', 'entropy_scaled', 'epoch_idx', 'recording_id', 'rel_alpha', 'rel_beta', 'rel_delta', 'rel_theta', 'rms', 'rms_scaled', 'std', 'std_scaled', 't_center_sec', 't_end_sec', 't_start_sec', 'theta', 'theta_alpha_ratio', 'theta_scaled', 'var', 'var_scaled', 'y_seizure']

New CHB24 columns (24): ['alpha', 'beta', 'dataset', 'delta', 'delta_beta_ratio', 'entropy', 'epoch_idx', 'next_seizure_start_sec', 'recording_id', 'rel_alpha', 'rel_beta', 'rel_delta', 'rel_theta', 'rms', 'std', 'subject_id', 't_center_sec', 't_end_sec', 't_start_sec', 'theta', 'theta_alpha_ratio', 'time_to_next_seizure_sec', 'var', 'y_seizure']

qEEG feature columns identical? True

Columns only in existing CHB01 table: ['alpha_scaled', 'beta_scaled', 'delta_scaled', 'entropy_scaled', 'rms_scaled', 'std_scaled', 'theta_scaled', 'var_s


  y_seizure                existing=int64      new=int64      compatible=True

Missing values in new table (excluding intentional NaN forecasting cols): 0
Infinite values in new table: 0

[OK] CHB24 table is schema- and dtype-compatible with the existing CHB01 table


In [12]:
print("\n" + "=" * 60)
print("STEP 11: FEATURE SANITY (count/mean/std/min/max/NaN/Inf per feature)")
print("=" * 60)

sanity_rows = []
for col in qeeg_feature_cols:
    s = feat_chb24_all[col]
    sanity_rows.append({
        "feature": col,
        "count": s.count(),
        "mean": s.mean(),
        "std": s.std(),
        "min": s.min(),
        "max": s.max(),
        "n_nan": int(s.isna().sum()),
        "n_inf": int(np.isinf(s).sum()),
    })

sanity_df = pd.DataFrame(sanity_rows)
print(sanity_df.to_string(index=False))

corrupted = sanity_df[(sanity_df["n_nan"] > 0) | (sanity_df["n_inf"] > 0)]
if len(corrupted) > 0:
    raise ValueError(f"Corrupted qEEG features found:\n{corrupted}")

# Physiological sanity: same non-negativity / [0,1] checks as EEG_preprocessing.ipynb
band_cols = ["delta", "theta", "alpha", "beta"]
rel_cols = ["rel_delta", "rel_theta", "rel_alpha", "rel_beta"]
nonneg_cols = ["std", "var", "rms"] + band_cols

for col in nonneg_cols:
    assert (feat_chb24_all[col] >= 0).all(), f"{col}: negative values found"
for col in rel_cols:
    assert (feat_chb24_all[col] >= -1e-9).all() and (feat_chb24_all[col] <= 1 + 1e-9).all(), \
        f"{col}: outside [0,1]"
assert np.isfinite(feat_chb24_all["entropy"]).all(), "entropy: non-finite values"

print("\n[OK] No NaN/Inf in qEEG features; band powers >= 0; relative powers in [0,1]")



STEP 11: FEATURE SANITY (count/mean/std/min/max/NaN/Inf per feature)


          feature  count         mean          std          min          max  n_nan  n_inf
              std   1440 5.456623e-05 3.697878e-05 2.310778e-05 2.516343e-04      0      0
              var   1440 4.343954e-09 7.445016e-09 5.339694e-10 6.331984e-08      0      0
              rms   1440 5.456641e-05 3.697878e-05 2.310778e-05 2.516355e-04      0      0
            delta   1440 2.826005e-09 4.840472e-09 2.430250e-10 4.681671e-08      0      0
            theta   1440 6.049946e-10 1.387264e-09 6.018294e-11 9.928206e-09      0      0
            alpha   1440 1.993335e-10 5.417813e-10 1.363943e-11 4.511005e-09      0      0
             beta   1440 2.934528e-10 5.836097e-10 1.296004e-11 4.608193e-09      0      0
        rel_delta   1440 7.546913e-01 9.589515e-02 2.735824e-01 9.281163e-01      0      0
        rel_theta   1440 1.457851e-01 6.381568e-02 4.810904e-02 4.677439e-01      0      0
        rel_alpha   1440 3.416728e-02 1.883907e-02 5.456696e-03 1.325166e-01      0      0

In [13]:
print("\n" + "=" * 60)
print("STEP 12: CHRONOLOGY VALIDATION")
print("=" * 60)

for recording_id, group in feat_chb24_all.groupby("recording_id"):
    idx = group["epoch_idx"].to_numpy()
    t_start = group["t_start_sec"].to_numpy()
    t_end = group["t_end_sec"].to_numpy()

    assert np.all(np.diff(idx) == 1), f"{recording_id}: epoch_idx not monotonic +1"
    assert np.all(np.diff(t_start) > 0), f"{recording_id}: t_start_sec not strictly increasing"
    assert np.all(t_end > t_start), f"{recording_id}: t_end_sec <= t_start_sec found"

    n_epochs = len(group)
    max_t_end = t_end.max()
    print(f"  {recording_id:14s}: {n_epochs:4d} epochs, t_start=[0, {t_start.max():.0f}]s, "
          f"max t_end={max_t_end:.0f}s")

print("\n[OK] epoch_idx monotonic (+1), t_start_sec strictly increasing, t_end_sec > t_start_sec "
      "for every recording")

# Recording-boundary check: max epoch t_end must not exceed the recording's
# actual EDF duration (no epoch crosses into another recording).
print("\nRecording-boundary check (epoch t_end vs actual EDF duration):")
for rec_id in chb24_recording_ids:
    recording_id = f"chb_s{SUBJECT_ID:02d}_r{rec_id.split('_')[1]}"
    group = feat_chb24_all[feat_chb24_all["recording_id"] == recording_id]
    max_t_end = group["t_end_sec"].max()
    actual_duration = edf_metadata[rec_id]["duration_sec"]
    ok = max_t_end <= actual_duration
    print(f"  {recording_id:14s}: max_t_end={max_t_end:7.1f}s <= edf_duration={actual_duration:7.1f}s -> {ok}")
    assert ok, f"{recording_id}: epoch t_end exceeds recording duration -- boundary crossed"

print("\n[OK] No epoch crosses its own EDF recording boundary")



STEP 12: CHRONOLOGY VALIDATION
  chb_s24_r01   :  120 epochs, t_start=[0, 3570]s, max t_end=3600s
  chb_s24_r03   :  120 epochs, t_start=[0, 3570]s, max t_end=3600s
  chb_s24_r04   :  120 epochs, t_start=[0, 3570]s, max t_end=3600s
  chb_s24_r06   :  120 epochs, t_start=[0, 3570]s, max t_end=3600s
  chb_s24_r07   :  120 epochs, t_start=[0, 3570]s, max t_end=3600s
  chb_s24_r09   :  120 epochs, t_start=[0, 3570]s, max t_end=3600s
  chb_s24_r11   :  120 epochs, t_start=[0, 3570]s, max t_end=3600s
  chb_s24_r13   :  120 epochs, t_start=[0, 3570]s, max t_end=3600s
  chb_s24_r14   :  120 epochs, t_start=[0, 3570]s, max t_end=3600s
  chb_s24_r15   :  120 epochs, t_start=[0, 3570]s, max t_end=3600s
  chb_s24_r17   :  120 epochs, t_start=[0, 3570]s, max t_end=3600s
  chb_s24_r21   :  120 epochs, t_start=[0, 3570]s, max t_end=3600s

[OK] epoch_idx monotonic (+1), t_start_sec strictly increasing, t_end_sec > t_start_sec for every recording

Recording-boundary check (epoch t_end vs actual EDF du

In [14]:
print("\n" + "=" * 60)
print("STEP 13: SEIZURE MAPPING VALIDATION")
print("=" * 60)

# Interval-intersection convention (identical to EEG_preprocessing.ipynb):
# epoch marked positive iff epoch_start < seizure_end AND epoch_end > seizure_start.
print("Labeling convention: epoch positive iff [t_start_sec, t_end_sec) intersects")
print("[seizure_start, seizure_end) -- i.e. epoch_start < seizure_end AND epoch_end > seizure_start.\n")

print(f"{'recording':14s} {'onset':>7s} {'end':>7s} {'epochs_before_onset':>20s} {'overlapping_epochs':>19s}")
for recording_id, intervals in sorted(chb24_seizure_intervals_final.items()):
    group = feat_chb24_all[feat_chb24_all["recording_id"] == recording_id]
    for onset, end in intervals:
        n_before = int((group["t_end_sec"] <= onset).sum())
        overlap_mask = (group["t_start_sec"] < end) & (group["t_end_sec"] > onset)
        n_overlap = int(overlap_mask.sum())
        print(f"{recording_id:14s} {onset:7.0f} {end:7.0f} {n_before:20d} {n_overlap:19d}")

# Boundary-case illustration: pick the first seizure in the dataset and show
# the two epochs immediately surrounding its onset explicitly.
example_recording_id, example_intervals = sorted(chb24_seizure_intervals_final.items())[0]
example_onset, example_end = example_intervals[0]
example_group = feat_chb24_all[feat_chb24_all["recording_id"] == example_recording_id]
onset_epoch_idx = int(example_onset // EPOCH_DURATION_SEC)

print(f"\nBoundary illustration ({example_recording_id}, onset={example_onset:.0f}s, "
      f"end={example_end:.0f}s):")
for offset in (-1, 0, 1):
    ei = onset_epoch_idx + offset
    row = example_group[example_group["epoch_idx"] == ei]
    if len(row) == 0:
        continue
    row = row.iloc[0]
    is_positive = bool(row["y_seizure"])
    print(f"  epoch {ei:4d}: [{row['t_start_sec']:.0f}, {row['t_end_sec']:.0f})s -> "
          f"y_seizure={int(row['y_seizure'])} "
          f"({'intersects' if is_positive else 'does not intersect'} [{example_onset:.0f}, {example_end:.0f}))")

# Every positive epoch must genuinely intersect SOME annotated interval for
# its own recording (defends against off-by-one / mislabeling regressions).
print("\nVerifying every y_seizure=1 epoch genuinely intersects an annotated interval...")
n_checked = 0
for recording_id, intervals in chb24_seizure_intervals_final.items():
    pos = feat_chb24_all[(feat_chb24_all["recording_id"] == recording_id) &
                          (feat_chb24_all["y_seizure"] == 1)]
    for _, row in pos.iterrows():
        intersects_any = any(
            row["t_start_sec"] < end and row["t_end_sec"] > start
            for start, end in intervals
        )
        assert intersects_any, (
            f"{recording_id} epoch {row['epoch_idx']}: y_seizure=1 but does not "
            f"intersect any annotated interval {intervals}"
        )
        n_checked += 1
print(f"[OK] {n_checked} positive epochs all verified to genuinely intersect an annotated seizure interval")



STEP 13: SEIZURE MAPPING VALIDATION
Labeling convention: epoch positive iff [t_start_sec, t_end_sec) intersects
[seizure_start, seizure_end) -- i.e. epoch_start < seizure_end AND epoch_end > seizure_start.

recording        onset     end  epochs_before_onset  overlapping_epochs
chb_s24_r01        480     505                   16                   1
chb_s24_r01       2451    2476                   81                   2
chb_s24_r03        231     260                    7                   2
chb_s24_r03       2883    2908                   96                   1
chb_s24_r04       1088    1120                   36                   2
chb_s24_r04       1411    1438                   47                   1
chb_s24_r04       1745    1764                   58                   1
chb_s24_r06       1229    1253                   40                   2
chb_s24_r07         38      60                    1                   1
chb_s24_r09       1745    1764                   58                   1


chb_s24_r14       1939    1966                   64                   2
chb_s24_r15       3552    3569                  118                   1
chb_s24_r17       3515    3581                  117                   3
chb_s24_r21       2804    2872                   93                   3

Boundary illustration (chb_s24_r01, onset=480s, end=505s):
  epoch   15: [450, 480)s -> y_seizure=0 (does not intersect [480, 505))
  epoch   16: [480, 510)s -> y_seizure=1 (intersects [480, 505))
  epoch   17: [510, 540)s -> y_seizure=0 (does not intersect [480, 505))

Verifying every y_seizure=1 epoch genuinely intersects an annotated interval...


[OK] 28 positive epochs all verified to genuinely intersect an annotated seizure interval


In [15]:
print("\n" + "=" * 60)
print("CHB24 DATA PREPARATION -- EXECUTION REPORT")
print("=" * 60)

n_edfs = len(chb24_recording_ids)
n_seizure_recordings = sum(1 for v in chb24_seizure_intervals_final.values() if len(v) > 0)

print(f"""
Files:
  EDFs processed              : {n_edfs}
  Seizure-containing recordings: {n_seizure_recordings}
  Total seizure events         : {n_seizure_events_total}
  Total downloaded size        : {total_bytes / 1e6:.1f} MB

Preprocessing compatibility with existing CHB01 representation:
  epoch duration    : {EPOCH_DURATION_SEC}s (matches)
  epoch overlap     : {EPOCH_OVERLAP} (matches)
  sampling frequency: {chb24_sfreq} Hz (matches, metadata-derived)
  filter            : {FILTER_LOW_HZ}-{FILTER_HIGH_HZ} Hz (matches)
  channels          : {len(chb24_actual_channels)} (montage compared above)
  qEEG feature count: {len(qeeg_feature_cols)} (identical names/formulas, verbatim reuse)

Output:
  {output_parquet_path}
  rows       : {feat_chb24_all.shape[0]}
  columns    : {feat_chb24_all.shape[1]}
  recordings : {n_edfs}
  seizures   : {n_seizure_events_total}
  NaNs (excl. forecasting cols): {n_nan_new}
  Infs       : {n_inf_new}
""")

print("=" * 60)
print("READY FOR TIME-TO-SEIZURE TARGET CONSTRUCTION")
print("=" * 60)



CHB24 DATA PREPARATION -- EXECUTION REPORT

Files:
  EDFs processed              : 12
  Seizure-containing recordings: 12
  Total seizure events         : 16
  Total downloaded size        : 508.8 MB

Preprocessing compatibility with existing CHB01 representation:
  epoch duration    : 30.0s (matches)
  epoch overlap     : 0.0 (matches)
  sampling frequency: 256.0 Hz (matches, metadata-derived)
  filter            : 0.5-40.0 Hz (matches)
  channels          : 23 (montage compared above)
  qEEG feature count: 14 (identical names/formulas, verbatim reuse)

Output:
  /home/evgenija/projects/EEG_timeseries_project/outputs/features_chb24_forecasting.parquet
  rows       : 1440
  columns    : 24
  recordings : 12
  seizures   : 16
  NaNs (excl. forecasting cols): 0
  Infs       : 0

READY FOR TIME-TO-SEIZURE TARGET CONSTRUCTION
